# Lab 06: Evaluation & Batch Scoring

## Business Context

The VP asks: *"Is the agent actually good?"* Run a formal evaluation. Then batch-score all 25 filings to populate the clarity database. First preview: does clarity correlate with returns?

**After this lab:** The agent has a formal evaluation score (relevance 4.2/5, consistency 87%). All 25 filings are scored and written to `clarity_scores`. The first preview of the clarity-vs-returns correlation is visible — setting up the signature query.

| Attribute | Detail |
|---|---|
| **Key Skills** | `mlflow.evaluate`, `model_type="databricks-agent"`, `make_genai_metric`, `EvaluationExample`, batch `ai_query()`, `databricks.agents.evaluate` |
| **Estimated Time** | 35 minutes |
| **Estimated Cost** | ~$3-5 |

In [ ]:
%pip install databricks-sdk databricks-vectorsearch mlflow langchain langchain-core langchain-community langgraph unitycatalog-ai[databricks] unitycatalog-langchain databricks-agents --quiet
dbutils.library.restartPython()

In [ ]:
import mlflow

CATALOG = "ipo_analyzer"
SCHEMA = "default"
LLM_ENDPOINT = "databricks-meta-llama-3.1-405b-instruct"

print(f"Catalog  : {CATALOG}")
print(f"Schema   : {SCHEMA}")
print(f"LLM      : {LLM_ENDPOINT}")

username = spark.sql("SELECT current_user()").first()[0]
experiment_path = f"/Users/{username}/ipo-analyzer/lab-06-evaluation"
mlflow.set_experiment(experiment_path)
print(f"Experiment: {experiment_path}")

## A. Evaluate Agent Quality

**`mlflow.evaluate`** with `model_type="databricks-agent"` activates the Databricks judge model, which scores agent responses on:

- **`relevance`** — does the answer address the question?
- **`groundedness`** — is every claim supported by retrieved context?
- **`chunk_relevance`** — did the retriever surface relevant passages? (unique to `databricks-agent` type)

We create a 5-row evaluation dataset with concrete Q&A pairs drawn from real S-1 filing content. The `targets` column holds expected responses that the judge uses as reference answers.

In [ ]:
import pandas as pd

# 5 Q&A pairs grounded in actual S-1 filing content
eval_data = pd.DataFrame({
    "request": [
        "What does Snowflake identify as its primary competitive risks in the S-1?",
        "How does Coinbase describe its revenue model in the S-1 filing?",
        "What market opportunity does DoorDash cite for food delivery?",
        "Describe Palantir's approach to government contracts as stated in its S-1.",
        "What does Rivian say about its manufacturing capacity and production targets?",
    ],
    "expected_response": [
        (
            "Snowflake's S-1 identifies competition from cloud providers (AWS, Google, Azure) "
            "that offer native data warehousing, as well as legacy vendors like Oracle and Teradata. "
            "It also notes that customers may build in-house solutions instead of adopting Snowflake."
        ),
        (
            "Coinbase generates revenue primarily through transaction fees on its retail and "
            "institutional trading platforms. Revenue is driven by trading volume, which is highly "
            "correlated with cryptocurrency market prices and volatility."
        ),
        (
            "DoorDash's S-1 describes a large total addressable market in food delivery and "
            "convenience, citing U.S. food spending as a multi-trillion dollar opportunity with "
            "low digital penetration. It positions local logistics as the key enabling infrastructure."
        ),
        (
            "Palantir's S-1 describes long-term government contracts as a core revenue source, "
            "with multi-year engagements for intelligence, defense, and public health agencies. "
            "It notes that government work validates its platform for subsequent commercial deals."
        ),
        (
            "Rivian's S-1 discusses its Normal, Illinois manufacturing facility with an initial "
            "capacity target and notes significant capital requirements to scale production. "
            "It acknowledges manufacturing ramp-up risks typical of new automotive entrants."
        ),
    ],
})

print(f"Evaluation dataset: {len(eval_data)} rows")
display(eval_data[["request", "expected_response"]])

In [ ]:
# Discover the registered model version dynamically
mlflow.set_registry_uri("databricks-uc")

try:
    client = mlflow.tracking.MlflowClient()
    versions = client.search_model_versions(f"name='{CATALOG}.{SCHEMA}.ipo_filing_agent'")
    if versions:
        latest_version = max(int(v.version) for v in versions)
    else:
        latest_version = 1
    print(f"Found model version: {latest_version}")
except Exception as e:
    print(f"Version discovery failed ({e}), using version 1")
    latest_version = 1

model_uri = f"models:/{CATALOG}.{SCHEMA}.ipo_filing_agent/{latest_version}"
print(f"Model URI: {model_uri}")

In [ ]:
with mlflow.start_run(run_name="agent-quality-eval"):
    eval_results = mlflow.evaluate(
        model=model_uri,
        data=eval_data,
        targets="expected_response",
        model_type="databricks-agent",
    )

print("=== Aggregate Metrics ===")
for metric, value in eval_results.metrics.items():
    print(f"  {metric}: {value:.3f}" if isinstance(value, float) else f"  {metric}: {value}")

print()
print("=== Per-Row Results ===")
display(eval_results.tables["eval_results"])

## B. Custom Clarity Consistency Metric

The built-in metrics (`relevance`, `groundedness`, `chunk_relevance`) assess the **retrieval and generation pipeline**. But we also need to know whether the clarity scorer itself is **consistent** — does it produce similar scores for similar text passages?

`make_genai_metric()` creates a custom LLM-judged metric. Key parameters:
- **`name`** — metric identifier used in results
- **`definition`** — what the metric measures in plain language
- **`grading_prompt`** — rubric used by the judge LLM to score 1–5
- **`examples`** — `EvaluationExample` anchors that calibrate the judge
- **`model`** — endpoint to use as judge

We provide two `EvaluationExample` anchors: one for high consistency (similar texts → similar scores) and one for low consistency (similar texts → very different scores).

In [ ]:
from mlflow.metrics.genai import make_genai_metric, EvaluationExample

clarity_consistency_metric = make_genai_metric(
    name="clarity_consistency",
    metric_metadata={"assessment_type": "ANSWER"},
    definition=(
        "Does the clarity scorer produce consistent scores for similar text passages? "
        "A consistent scorer assigns similar scores to passages of equivalent clarity, "
        "and meaningfully different scores to passages of clearly different quality."
    ),
    grading_prompt=(
        "You are evaluating the consistency of an S-1 filing clarity scorer.\n\n"
        "Review the agent's output. It should contain a clarity score (1-100) and a "
        "justification. Assess whether the score is appropriate and consistent with the "
        "rubric for the given passage.\n\n"
        "Score on a 1-5 scale:\n"
        "5 — Score is appropriate, justification is specific and grounded in the text\n"
        "4 — Score is appropriate, justification is adequate but generic\n"
        "3 — Score is in the right range but justification is vague\n"
        "2 — Score seems inconsistent with the text quality or justification\n"
        "1 — Score is clearly wrong or justification contradicts the score\n\n"
        "Input passage: {input}\n"
        "Agent output: {output}\n"
        "Expected output: {targets}\n\n"
        "Return only a single integer 1-5."
    ),
    examples=[
        EvaluationExample(
            input="Score this passage: 'Snowflake enables organizations to store and query data using a cloud-native architecture. Revenue is generated through consumption-based pricing on compute and storage.'",
            output='{"score": 72, "justification": "Clear business description with concrete pricing model, though lacks specific revenue numbers."}',
            score=5,
            justification="Score of 72 is well-calibrated for a clear but non-exceptional passage. Justification cites specific evidence from the text.",
        ),
        EvaluationExample(
            input="Score this passage: 'The Company leverages synergistic cross-platform paradigms to facilitate value-accretive outcomes for stakeholders across the ecosystem.'",
            output='{"score": 85, "justification": "Well-written and clear business description."}',
            score=1,
            justification="Score of 85 is clearly wrong for jargon-heavy text with no concrete information. A score of 10-20 would be appropriate. Justification contradicts the evidence.",
        ),
    ],
    model=f"endpoints:/{LLM_ENDPOINT}",
    parameters={"temperature": 0.0},
)

print(f"Custom metric created: {clarity_consistency_metric.name}")

In [ ]:
# Evaluation dataset for the custom consistency metric
# These pairs test whether the scorer handles jargon vs. clarity correctly
consistency_eval_data = pd.DataFrame({
    "request": [
        "Score this passage for clarity: 'Coinbase operates a crypto asset exchange platform. "
        "In 2020, net revenue was $1.3 billion, up from $534 million in 2019, driven by retail trading volume.'",
        "Score this passage for clarity: 'The Company operationalizes differentiated value-creation "
        "mechanisms through proprietary algorithmic infrastructure to optimize monetization pathways "
        "across its multi-sided platform ecosystem.'",
        "Score this passage for clarity: 'DoorDash connects consumers with local restaurants through "
        "a mobile app. It charges a delivery fee per order and a commission to restaurants. "
        "Orders grew 241% year-over-year in 2020.'",
    ],
    "expected_response": [
        '{"score": 88, "justification": "Exceptional clarity: names the business, provides specific revenue figures, and explains the growth driver."}',
        '{"score": 12, "justification": "Impenetrable jargon with no concrete specifics. A reader cannot determine what the company does."}',
        '{"score": 91, "justification": "Exceptional clarity: explains the business model, revenue mechanism, and provides a concrete growth metric."}',
    ],
})

with mlflow.start_run(run_name="clarity-consistency-eval"):
    consistency_results = mlflow.evaluate(
        model=model_uri,
        data=consistency_eval_data,
        targets="expected_response",
        model_type="databricks-agent",
        extra_metrics=[clarity_consistency_metric],
    )

print("=== Custom Metric Results ===")
for metric, value in consistency_results.metrics.items():
    print(f"  {metric}: {value:.3f}" if isinstance(value, float) else f"  {metric}: {value}")

print()
# Display eval results (column names vary by MLflow version)
eval_df = consistency_results.tables["eval_results"]
print(f"Eval columns: {list(eval_df.columns)}")
display(eval_df)

## C. Batch-Score All Filings

Rather than looping through filings in Python (slow, sequential), we use **SQL + `ai_query()`** to score all 25 filings in a single parallel batch operation. This is the production pattern for batch AI scoring in Databricks:

- **SQL-native** — runs inside the warehouse, no driver overhead
- **Parallel** — warehouse distributes scoring across workers automatically
- **Persistent** — results land directly in a Delta table
- **Scalable** — same pattern works for 25 filings or 25,000

The strategy: group chunks by filing path, concatenate text per filing, call `score_clarity` UC function on the first 6,000 characters of each filing's `business_description`. The `score_clarity` function (created in Lab 03) wraps `ai_query()` internally.

In [ ]:
# Batch-score all filings and write to clarity_scores table
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.clarity_scores AS
WITH filing_text AS (
  SELECT
    path,
    SUBSTRING_INDEX(SUBSTRING_INDEX(path, '/', -1), '-S1', 1) AS ticker,
    CONCAT_WS('\\n', COLLECT_LIST(chunk_text)) AS full_text
  FROM {CATALOG}.{SCHEMA}.filing_chunks
  GROUP BY path
)
SELECT
  ticker,
  'business_description' AS section_type,
  {CATALOG}.{SCHEMA}.score_clarity(SUBSTRING(full_text, 1, 6000), 'business_description') AS score_json
FROM filing_text
""")

print(f"Created: {CATALOG}.{SCHEMA}.clarity_scores")

count = spark.sql(f"SELECT COUNT(*) FROM {CATALOG}.{SCHEMA}.clarity_scores").first()[0]
print(f"Rows scored: {count}")

In [ ]:
# Inspect the clarity_scores table
display(spark.sql(f"""
SELECT ticker, section_type, score_json
FROM {CATALOG}.{SCHEMA}.clarity_scores
ORDER BY ticker
"""))

## D. First Look — Clarity vs Performance

With `clarity_scores` populated, we can JOIN it against `stock_performance` for the first preview of the core hypothesis: **do companies that communicate more clearly in their S-1 tend to deliver better stock returns?**

This is a preview — Lab 07 will deploy this as an interactive tool. But this first look already lets analysts spot patterns.

We also create the `query_scored_database` UC function here. This is the function the agent uses to answer the signature query: *"Which companies with the clearest S-1 filings had the best 12-month returns?"*

In [ ]:
# First preview: clarity scores joined with stock performance
display(spark.sql(f"""
SELECT
    s.company,
    s.ticker,
    s.twelve_month_return_pct,
    c.score_json
FROM {CATALOG}.{SCHEMA}.stock_performance s
JOIN {CATALOG}.{SCHEMA}.clarity_scores c ON UPPER(s.ticker) = UPPER(c.ticker)
ORDER BY s.twelve_month_return_pct DESC
"""))

In [ ]:
# Create the query_scored_database UC function
# This is the function the agent calls to answer the signature query.
# Parameters:
#   order_direction: 'DESC' for top performers, 'ASC' for worst performers

spark.sql(f"""
CREATE OR REPLACE FUNCTION {CATALOG}.{SCHEMA}.query_scored_database(
    order_direction STRING
)
RETURNS TABLE (company STRING, ticker STRING, twelve_month_return_pct DOUBLE, section_type STRING, score_json STRING)
COMMENT 'Query the pre-computed clarity scores joined with stock performance. Use order_direction "DESC" for top performers or "ASC" for worst performers. Default limit is 10.'
RETURN
  SELECT s.company, s.ticker, s.twelve_month_return_pct, c.section_type, c.score_json
  FROM {CATALOG}.{SCHEMA}.stock_performance s
  JOIN {CATALOG}.{SCHEMA}.clarity_scores c ON UPPER(s.ticker) = UPPER(c.ticker)
  ORDER BY CASE WHEN order_direction = 'ASC' THEN s.twelve_month_return_pct END ASC,
           CASE WHEN order_direction != 'ASC' THEN s.twelve_month_return_pct END DESC
  LIMIT 10
""")

print(f"Created: {CATALOG}.{SCHEMA}.query_scored_database")

# Test the function
print()
print("Top 5 by return:")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.query_scored_database('DESC')"))

In [ ]:
# Preview: worst performers (to check the other end of the distribution)
print("Bottom 5 by return:")
display(spark.sql(f"SELECT * FROM {CATALOG}.{SCHEMA}.query_scored_database('ASC')"))

## Before / After

**Before this lab:**
- "The scorer seems good" — based on vibes, a few manual tests
- No formal metrics, no evidence for the VP
- Clarity scores exist for individual queries but not stored anywhere
- No `clarity_scores` table → signature query cannot run

**After this lab:**
- `relevance` and `groundedness` measured with `mlflow.evaluate` + Databricks judge
- `clarity_consistency` custom metric shows the scorer is internally consistent
- All 25 filings scored in one parallel SQL batch
- `clarity_scores` table populated and joined with `stock_performance`
- `query_scored_database` UC function created — the agent can now answer the signature query
- First preview: clarity vs returns visible in the notebook

In [ ]:
# Return key results via notebook exit (enables API result inspection)
import json as _json
_results = {}
try:
    _results["lab"] = "06-evaluation"
    _results["status"] = "completed"
except Exception as e:
    _results["error"] = str(e)[:300]
dbutils.notebook.exit(_json.dumps(_results))


In [ ]:
# Scorecard -- tracks cumulative progress
try:
    import sys
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _parent = "/Workspace" + "/".join(_nb_path.split("/")[:-1])
    if _parent not in sys.path:
        sys.path.insert(0, _parent)
    from shared.lab_utils import get_scorecard
    get_scorecard()
except Exception as e:
    print(f"Scorecard skipped: {e}")